In [ ]:
import pandas as pd
import unicodedata as unicode

# ==============================================================================
# 1. FUNÇÃO PARA REMOVER ACENTOS
# ==============================================================================
# Recebe qualquer texto e remove os acentos sem alterar as letras base.
def remover_acentos(texto):
    if pd.isna(texto):
        return texto
    return unicode.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('ASCII')

# ==============================================================================
# 2. FUNÇÃO PARA FORMATAR VALORES FINANCEIROS (MOEDA)
# ==============================================================================
# O Supabase (PostgreSQL) exige valores numéricos no formato internacional:
# - Sem o símbolo 'R$' e sem espaços
# - Sem pontos de milhar (ex: 1.000 vira 1000)
# - Com PONTO (.) no lugar da VÍRGULA (,) para separar as casas decimais (ex: 1250,50 vira 1250.50)
def limpar_moeda(valor):
    if pd.isna(valor):
        return valor
    valor_str = str(valor).replace('R$', '').strip()  # Remove o 'R$' e espaços nas pontas
    valor_str = valor_str.replace('.', '')            # Remove o ponto de milhar (ex: 1.250 -> 1250)
    valor_str = valor_str.replace(',', '.')           # Troca a vírgula decimal por ponto (ex: 1250,50 -> 1250.50)
    return valor_str

# ==============================================================================
# 3. LEITURA DO ARQUIVO CSV ORIGINAL
# ==============================================================================
df = pd.read_csv('../base_csv/original/Produto.csv', encoding='latin1', sep=';')

# ==============================================================================
# 4. TRATAMENTO DO CABEÇALHO (NOMES DAS COLUNAS)
# ==============================================================================
# Transforma todas as colunas para minúsculas e sem acento (ex: 'CUSTO' -> 'custo')
df.columns = [remover_acentos(col).lower() for col in df.columns]

# ==============================================================================
# 5. TRATAMENTO DA COLUNA DE DATA ('registro')
# ==============================================================================
# Converte a data brasileira (dia/mês/ano) para o formato internacional AAAA-MM-DD exigido pelo Supabase
if 'registro' in df.columns:
    df['registro'] = pd.to_datetime(df['registro'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')

# ==============================================================================
# 6. TRATAMENTO DE VALORES MONETÁRIOS (SE HOUVER VALOR MONETÁRIO)
# ==============================================================================
# Limpa as colunas 'custo' e 'venda' convertendo a formatação de moeda brasileira para o padrão decimal
for col in ['custo', 'venda']:
    if col in df.columns:
        df[col] = df[col].apply(limpar_moeda)

# ==============================================================================
# 7. TRATAMENTO DO CONTEÚDO DE TODAS AS DEMAIS CÉLULAS
# ==============================================================================
# Aplica a remoção de acentos em toda a tabela
df = df.map(remover_acentos)

# ==============================================================================
# 8. SALVANDO O ARQUIVO TRATADO
# ==============================================================================
# Salva o arquivo em UTF-8 com marcador BOM para importação perfeita no Supabase e visualização no Excel
df.to_csv('../base_csv/tratado/contrucao_tratado.csv', index=False, sep=';', encoding='utf-8-sig')

print("Tratamento concluído! Arquivo salvo em base_csv/tratado/")